# PARSEC and MIST Source-Isochrone Systematics Smoke Test

This notebook is a smoke-test draft for the public PARSEC/MIST source-forward example. It compares the default PARSEC source prior with MIST solar-scaled, MIST alpha-enhanced, and a PARSEC-primary/MIST-alpha mixture.

The goal is not to make a final science plot here. The checks below verify that the configurations run, produce finite source-property columns, and expose useful summary diagnostics for a later public example.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / 'CMakeLists.txt').exists() and (path / 'build').exists():
            return path
    return start

repo_root = find_repo_root(Path.cwd().resolve())
build_dir = repo_root / 'build'
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens


## Shared Setup

All smoke runs use the same sight line, seed, source-forward mass bounds, and Roman photometry. Source-selection cuts are intentionally omitted here so the smoke test stays fast; the public example can add apparent-magnitude cuts and extinction.


In [ ]:
N_SIMU = 5
SEED = 2026
SOURCE_COLUMNS = [
    'D_S',
    'MH_S',
    'M_S_ini',
    'M_S',
    'R_S',
    'teff_S',
    'logg_S',
    'theta_S',
    'M_F146mag_S',
]


def as_dataframe(result):
    return pd.DataFrame(result.to_numpy(), columns=result.columns)


def base_config():
    cfg = genulens.Config(l=1.0, b=-3.9, n_simu=N_SIMU, seed=SEED)
    cfg.forward_source.enabled = 1
    cfg.forward_source.photometry = 'roman'
    cfg.forward_source.min_initial_mass_msun = 0.1
    cfg.forward_source.max_initial_mass_msun = 2.0
    return cfg


## Model Configurations

The smoke run keeps only two configurations: the default PARSEC solar baseline and a MIST alpha-enhanced primary table. The helper functions also show how to configure MIST solar and a PARSEC-primary/MIST-alpha mixture for the later public example.


In [ ]:
def parsec_baseline_config():
    cfg = base_config()
    cfg.forward_source.isochrone_family = 'parsec'
    cfg.forward_source.isochrone_abundance = 'solar_scaled'
    return cfg


def mist_primary_config(abundance):
    cfg = base_config()
    cfg.forward_source.isochrone_family = 'mist'
    cfg.forward_source.isochrone_abundance = abundance
    cfg.forward_source.isochrone_alpha_fe = 0.4 if abundance == 'alpha_enhanced' else 0.0
    return cfg


def parsec_mist_alpha_mixture_config(fraction=0.5):
    cfg = base_config()
    cfg.forward_source.isochrone_model = 'alpha_mixture'
    cfg.forward_source.isochrone_family = 'parsec'
    cfg.forward_source.isochrone_abundance = 'solar_scaled'
    cfg.forward_source.secondary_isochrone_family = 'mist'
    cfg.forward_source.secondary_isochrone_abundance = 'alpha_enhanced'
    cfg.forward_source.secondary_isochrone_alpha_fe = 0.4
    cfg.forward_source.alpha_enhanced_fraction = fraction
    return cfg


configs = {
    'parsec_solar': parsec_baseline_config(),
    'mist_alpha': mist_primary_config('alpha_enhanced'),
}


## Run and Validate

These assertions are deliberately simple: both configurations should return `N_SIMU` rows, include the source-property columns, and keep the checked source properties finite.


In [ ]:
frames = {}
for name, cfg in configs.items():
    result = genulens.simulate(cfg)
    df = as_dataframe(result)
    assert len(df) == N_SIMU, name
    for column in SOURCE_COLUMNS:
        assert column in df.columns, (name, column)
    assert np.isfinite(df[SOURCE_COLUMNS].to_numpy()).all(), name
    frames[name] = df

pd.DataFrame({
    'run': list(frames),
    'rows': [len(df) for df in frames.values()],
    'median_D_S': [df['D_S'].median() for df in frames.values()],
    'median_teff_S': [df['teff_S'].median() for df in frames.values()],
    'median_R_S': [df['R_S'].median() for df in frames.values()],
    'median_M_F146mag_S': [df['M_F146mag_S'].median() for df in frames.values()],
})


## Weighted Quantile Summary

The smoke test records a compact summary that is useful when turning this into a public example. The values should not be interpreted as final science numbers because `N_SIMU` is intentionally small.


In [ ]:
def weighted_quantile(values, weights, quantiles):
    values = np.asarray(values)
    weights = np.asarray(weights)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(quantiles, cdf, values)


summary_rows = []
for name, df in frames.items():
    weights = df['wtj'].to_numpy()
    for column in SOURCE_COLUMNS:
        q16, q50, q84 = weighted_quantile(df[column].to_numpy(), weights, [0.16, 0.50, 0.84])
        summary_rows.append({'run': name, 'column': column, 'q16': q16, 'q50': q50, 'q84': q84})

summary = pd.DataFrame(summary_rows)
summary


## Quick Diagnostic Plot

This plot is intentionally lightweight. It checks that the source-property distributions can be compared visually before promoting the workflow to `examples/`.


In [ ]:
plot_columns = ['D_S', 'teff_S', 'R_S', 'M_F146mag_S']
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, column in zip(axes.flat, plot_columns):
    for name, df in frames.items():
        ax.hist(df[column], bins=30, weights=df['wtj'], histtype='step', density=True, label=name)
    ax.set_xlabel(column)
    ax.set_ylabel('weighted density')
axes[0, 0].legend(fontsize=8)
fig.suptitle('PARSEC and MIST source-property smoke comparison')
fig.tight_layout()
plt.show()


## Guardrail: PARSEC Alpha Is Unsupported

PARSEC alpha-enhanced tables are not part of this workflow. The API should fail loudly rather than constructing a fake table path.


In [ ]:
spec = genulens.IsochroneLibrarySpec()
spec.family = 'parsec'
spec.photometry = 'roman'
spec.abundance = 'alpha_enhanced'
try:
    genulens.default_isochrone_table_path(spec)
except RuntimeError as exc:
    assert 'PARSEC/CMD alpha-enhanced' in str(exc)
else:
    raise AssertionError('PARSEC alpha unexpectedly resolved to a table path')
